In [1]:
# ===============================
# 0. Dependencies & Setup
# ===============================
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install -q torch_geometric matplotlib tqdm scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import matplotlib.pyplot as plt
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH     = '/content/drive/MyDrive/Honors/'   # UPDATE THIS
TEXT_FEAT     = 'text_features_imp.npy'
AUDIO_FEAT    = 'audio_features_imp.npy'
VIDEO_FEAT    = 'video_features_imp.npy'
CONTEXT_FEAT  = 'context_features_imp.npy'
LABELS_FEAT   = 'labels_imp.npy'

BATCH_SIZE    = 16
EPOCHS        = 100
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 1e-4
PATIENCE      = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00
Using device: cpu


In [2]:
# ===============================
# 1. Load Features
# ===============================
from google.colab import drive
drive.mount('/content/drive')

print("Loading All Features (Text, Audio, Video, Context)...")

t_features = np.load(os.path.join(BASE_PATH, TEXT_FEAT))
a_features = np.load(os.path.join(BASE_PATH, AUDIO_FEAT))
v_features = np.load(os.path.join(BASE_PATH, VIDEO_FEAT))
c_features = np.load(os.path.join(BASE_PATH, CONTEXT_FEAT))
labels     = np.load(os.path.join(BASE_PATH, LABELS_FEAT))

print(f"Loaded {len(labels)} samples.")
print(f"Dims: T={t_features.shape[1]}, A={a_features.shape[1]}, V={v_features.shape[1]}, C={c_features.shape[1]}")


Mounted at /content/drive
Loading All Features (Text, Audio, Video, Context)...
Loaded 690 samples.
Dims: T=768, A=768, V=2048, C=2816


In [3]:
# ===============================
# 2. Build Graph Dataset
# ===============================
data_list = []
for i in range(len(labels)):
    t = torch.tensor(t_features[i], dtype=torch.float).unsqueeze(0)
    a = torch.tensor(a_features[i], dtype=torch.float).unsqueeze(0)
    v = torch.tensor(v_features[i], dtype=torch.float).unsqueeze(0)
    c = torch.tensor(c_features[i], dtype=torch.float).unsqueeze(0)
    y = torch.tensor([labels[i]], dtype=torch.float)

    # Nodes: 0=T, 1=A, 2=V, 3=C
    edge_index = torch.tensor([
        [0,1],[1,0], [0,2],[2,0], [1,2],[2,1],  # Modality triangle
        [3,0],[0,3], [3,1],[1,3], [3,2],[2,3],  # Context star
        [0,0],[1,1],[2,2],[3,3]                 # Self loops
    ], dtype=torch.long).t().contiguous()

    data = Data(edge_index=edge_index, y=y)
    data.text    = t
    data.audio   = a
    data.video   = v
    data.context = c
    data.num_nodes = 4  # suppress PyG warning

    data_list.append(data)

print(f"Built {len(data_list)} graph samples.")


Built 690 graph samples.


In [4]:
# ===============================
# 3. Train/Val/Test Split
# ===============================
# First: train+val vs test
train_val, test = train_test_split(
    data_list, test_size=0.15, random_state=42, stratify=labels
)

# Then: train vs val
labels_train_val = np.array([d.y.item() for d in train_val])
train, val = train_test_split(
    train_val, test_size=0.1765, random_state=42, stratify=labels_train_val
)  # 0.1765 * 0.85 ≈ 0.15, so ~70/15/15

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test,  batch_size=BATCH_SIZE, shuffle=False)


Train: 482, Val: 104, Test: 104


In [5]:
# ===============================
# 4. SarcasmGATv2 v2 Model
# ===============================
class SarcasmGATv2(nn.Module):
    def __init__(self, t_dim, a_dim, v_dim, c_dim, hidden_dim=256, gat_dropout=0.6, fc_dropout=0.6):
        super().__init__()

        # Projections
        self.p_t = nn.Sequential(nn.Linear(t_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_a = nn.Sequential(nn.Linear(a_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_v = nn.Sequential(nn.Linear(v_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.p_c = nn.Sequential(nn.Linear(c_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

        # GATv2
        self.gat1 = GATv2Conv(hidden_dim, hidden_dim, heads=8, concat=True, dropout=gat_dropout)
        self.gat2 = GATv2Conv(hidden_dim * 8, hidden_dim, heads=1, concat=False, dropout=gat_dropout)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(fc_dropout),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        x_t = self.p_t(data.text)
        x_a = self.p_a(data.audio)
        x_v = self.p_v(data.video)
        x_c = self.p_c(data.context)

        batch_size = x_t.size(0)
        d = x_t.size(1)

        x = torch.zeros(batch_size * 4, d, device=DEVICE)
        x[0::4] = x_t
        x[1::4] = x_a
        x[2::4] = x_v
        x[3::4] = x_c

        base_edges = data.edge_index[:, :16]  # 16 edges per graph
        edge_index_list = []
        for i in range(batch_size):
            edge_index_list.append(base_edges + i * 4)
        edge_index = torch.cat(edge_index_list, dim=1).to(DEVICE)

        out = self.gat1(x, edge_index)
        out = F.gelu(out)
        out = self.gat2(out, edge_index)

        batch_vec = torch.arange(batch_size, device=DEVICE).view(-1, 1).repeat(1, 4).view(-1)
        out = global_mean_pool(out, batch_vec)

        logits = self.fc(out)
        return logits.squeeze(-1)


In [6]:
# ===============================
# 5. Optimizer, Scheduler, Loss
# ===============================
model = SarcasmGATv2(
    t_dim=t_features.shape[1],
    a_dim=a_features.shape[1],
    v_dim=v_features.shape[1],
    c_dim=c_features.shape[1],
    hidden_dim=256,
    gat_dropout=0.6,
    fc_dropout=0.6,
).to(DEVICE)

# Optionally separate weight decay for projection/GAT vs others
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(nd in name for nd in ["bias", "LayerNorm"]):
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)

criterion = nn.BCEWithLogitsLoss()


In [7]:
# ===============================
# 6. Train / Eval Functions
# ===============================
def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for data in loader:
        data = data.to(DEVICE)
        optimizer.zero_grad()

        logits = model(data)
        loss = criterion(logits, data.y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.y.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_preds.extend(preds)
        all_labels.extend(labels)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for data in loader:
        data = data.to(DEVICE)
        logits = model(data)

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return acc, f1, prec, rec, auc


In [8]:
# ===============================
# 7. Training Loop with Early Stopping
# ===============================
best_val_f1 = 0.0
best_state  = None
patience_counter = 0

train_losses, val_f1s = [], []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = train_one_epoch(train_loader)
    val_acc, val_f1, val_prec, val_rec, val_auc = evaluate(val_loader)

    scheduler.step(epoch)

    train_losses.append(train_loss)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} F1 {train_f1:.4f} | "
        f"Val Acc {val_acc:.4f} F1 {val_f1:.4f} Prec {val_prec:.4f} Rec {val_rec:.4f} AUC {val_auc:.4f} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print(f"Training complete. Best Val F1: {best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(BASE_PATH, "best_sarcasm_gatv2_full.pth"))
    print("Best model saved.")


Epoch 001 | Train Loss 0.6941 Acc 0.4793 F1 0.4591 | Val Acc 0.5000 F1 0.3333 Prec 0.2500 Rec 0.5000 AUC 0.7086 | LR 0.000049
Epoch 002 | Train Loss 0.6892 Acc 0.5498 F1 0.5474 | Val Acc 0.5000 F1 0.3333 Prec 0.2500 Rec 0.5000 AUC 0.7311 | LR 0.000045
Epoch 003 | Train Loss 0.6896 Acc 0.5332 F1 0.5327 | Val Acc 0.6058 F1 0.5486 Prec 0.7142 Rec 0.6058 AUC 0.7659 | LR 0.000040
Epoch 004 | Train Loss 0.6880 Acc 0.5436 F1 0.5407 | Val Acc 0.5673 F1 0.4676 Prec 0.7680 Rec 0.5673 AUC 0.7955 | LR 0.000033
Epoch 005 | Train Loss 0.6712 Acc 0.6100 F1 0.6094 | Val Acc 0.6058 F1 0.5673 Prec 0.6641 Rec 0.6058 AUC 0.7962 | LR 0.000025
Epoch 006 | Train Loss 0.6584 Acc 0.6307 F1 0.6180 | Val Acc 0.6635 F1 0.6462 Prec 0.7032 Rec 0.6635 AUC 0.8033 | LR 0.000017
Epoch 007 | Train Loss 0.6272 Acc 0.7012 F1 0.7008 | Val Acc 0.6635 F1 0.6492 Prec 0.6953 Rec 0.6635 AUC 0.8018 | LR 0.000010
Epoch 008 | Train Loss 0.6121 Acc 0.6888 F1 0.6842 | Val Acc 0.7308 F1 0.7258 Prec 0.7488 Rec 0.7308 AUC 0.8029 | LR 0

In [9]:
# ===============================
# 8. Final Test Evaluation
# ===============================
test_acc, test_f1, test_prec, test_rec, test_auc = evaluate(test_loader)
print(f"TEST | Acc {test_acc:.4f} F1 {test_f1:.4f} Prec {test_prec:.4f} Rec {test_rec:.4f} AUC {test_auc:.4f}")


TEST | Acc 0.7788 F1 0.7787 Prec 0.7798 Rec 0.7788 AUC 0.8147
